# Data Cleaning – Version 1

This notebook prepares the initial cleaned dataset for the ride cancellation prediction project.

Main tasks:
- load the raw dataset
- filter relevant booking statuses
- create a binary target variable
- remove identifier and leakage-prone columns
- check missing values and duplicates
- save the cleaned dataset

## Step 1: Load the raw dataset
In this step, the raw booking dataset is imported into Python for initial inspection and cleaning.

In [2]:
import pandas as pd

In [3]:
from google.colab import files
uploaded = files.upload()

Saving uber.csv to uber.csv


In [4]:
uploaded.keys()

dict_keys(['uber.csv'])

## Step 2: Inspect the dataset
The dataset is loaded and checked for its structure, size, column names, and booking status distribution.

In [5]:
df = pd.read_csv("uber.csv")

In [6]:
df.head()

,Date,Time,Booking ID,Booking Status,Customer ID,Vehicle Type,Pickup Location,Drop Location,Avg VTAT,Avg CTAT,...,Reason for cancelling by Customer,Cancelled Rides by Driver,Driver Cancellation Reason,Incomplete Rides,Incomplete Rides Reason,Booking Value,Ride Distance,Driver Ratings,Customer Rating,Payment Method
0,3/23/2024,12:29:38,"""CNR5884300""",No Driver Found,"""CID1982111""",eBike,Palam Vihar,Jhilmil,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,11/29/2024,18:01:39,"""CNR1326809""",Incomplete,"""CID4604802""",Go Sedan,Shastri Nagar,Gurgaon Sector 56,4.9,14.0,...,NaN,NaN,NaN,1.0,Vehicle Breakdown,237.0,5.73,NaN,NaN,UPI
2,8/23/2024,8:56:10,"""CNR8494506""",Completed,"""CID9202816""",Auto,Khandsa,Malviya Nagar,13.4,25.8,...,NaN,NaN,NaN,NaN,NaN,627.0,13.58,4.9,4.9,Debit Card
3,10/21/2024,17:17:25,"""CNR8906825""",Completed,"""CID2610914""",Premier Sedan,Central Secretariat,Inderlok,13.1,28.5,...,NaN,NaN,NaN,NaN,NaN,416.0,34.02,4.6,5.0,UPI
4,9/16/2024,22:08:00,"""CNR1950162""",Completed,"""CID9933542""",Bike,Ghitorni Village,Khan Market,5.3,19.6,...,NaN,NaN,NaN,NaN,NaN,737.0,48.21,4.1,4.3,UPI


In [7]:
df.shape

(150000, 21)

In [8]:
df.columns

Index(['Date', 'Time', 'Booking ID', 'Booking Status', 'Customer ID',
       'Vehicle Type', 'Pickup Location', 'Drop Location', 'Avg VTAT',
       'Avg CTAT', 'Cancelled Rides by Customer',
       'Reason for cancelling by Customer', 'Cancelled Rides by Driver',
       'Driver Cancellation Reason', 'Incomplete Rides',
       'Incomplete Rides Reason', 'Booking Value', 'Ride Distance',
       'Driver Ratings', 'Customer Rating', 'Payment Method'],
      dtype='object')

In [9]:
df["Booking Status"].value_counts()

,count
Booking Status,
Completed,93000
Cancelled by Driver,27000
No Driver Found,10500
Cancelled by Customer,10500
Incomplete,9000


## Step 3: Keep only relevant booking statuses
For the primary modelling stage, only the following booking outcomes are retained:
- Completed
- Cancelled by Driver
- Cancelled by Customer

The statuses "No Driver Found" and "Incomplete" are excluded to maintain a clear and consistent definition of ride cancellation.

In [10]:
keep_status = ["Completed", "Cancelled by Driver", "Cancelled by Customer"]
df1 = df[df["Booking Status"].isin(keep_status)].copy()

In [11]:
df1["Booking Status"].value_counts()

,count
Booking Status,
Completed,93000
Cancelled by Driver,27000
Cancelled by Customer,10500


## Step 4: Create the binary target variable
A new target variable called `target_cancelled` is created:
- 0 = Completed
- 1 = Cancelled by Driver or Cancelled by Customer

In [12]:
df1["target_cancelled"] = df1["Booking Status"].map({
    "Completed": 0,
    "Cancelled by Driver": 1,
    "Cancelled by Customer": 1
})

In [13]:
df1["target_cancelled"].value_counts()

,count
target_cancelled,
0,93000
1,37500


## Step 5: Remove identifier and leakage-prone variables
Identifier columns and variables that may introduce information leakage are removed before the modelling stage.

In [14]:
drop_cols = [
    "Booking ID",
    "Customer ID",
    "Cancelled Rides by Customer",
    "Reason for cancelling by Customer",
    "Cancelled Rides by Driver",
    "Driver Cancellation Reason",
    "Incomplete Rides",
    "Incomplete Rides Reason",
    "Driver Ratings",
    "Customer Rating"
]

df2 = df1.drop(columns=drop_cols)

In [15]:
df2.columns

Index(['Date', 'Time', 'Booking Status', 'Vehicle Type', 'Pickup Location',
       'Drop Location', 'Avg VTAT', 'Avg CTAT', 'Booking Value',
       'Ride Distance', 'Payment Method', 'target_cancelled'],
      dtype='object')

## Step 6: Check missing values
Missing values are checked to identify incomplete variables and to determine whether some columns should be excluded from the initial modelling dataset.

In [16]:
df2.isna().sum()

,0
Date,0
Time,0
Booking Status,0
Vehicle Type,0
Pickup Location,0
Drop Location,0
Avg VTAT,0
Avg CTAT,37500
Booking Value,37500
Ride Distance,37500


## Step 7: Remove variables with substantial missingness
The variables `Avg CTAT`, `Booking Value`, `Ride Distance`, and `Payment Method` are removed from the initial cleaned dataset because they contain a large number of missing values in the filtered sample.

In [17]:
df3 = df2.drop(columns=["Avg CTAT", "Booking Value", "Ride Distance", "Payment Method"])

## Step 8: Recheck missing values
After removing incomplete variables, the dataset is checked again to confirm that the remaining columns do not contain missing values.

In [18]:
df3.isna().sum()

,0
Date,0
Time,0
Booking Status,0
Vehicle Type,0
Pickup Location,0
Drop Location,0
Avg VTAT,0
target_cancelled,0


## Step 9: Check duplicate rows
The cleaned dataset is checked for fully duplicated records.

In [19]:
df3.duplicated().sum()

np.int64(0)

## Step 10: Save and export the cleaned dataset
The cleaned dataset is saved as `model_ready_v1.csv` for use in the next stage of the project, which is exploratory data analysis.

In [20]:
df3.to_csv("model_ready_v1.csv", index=False)

In [21]:
from google.colab import files
files.download("model_ready_v1.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>